# ⏱️ 实验二：优化前基线 —— 未融合推理

## 学习目标

1. 在 NPU / CPU 上运行未优化的 MobileNetV3 推理
2. 记录平均时延、吞吐、峰值显存与算子数量
3. 把结果保存到 results/baseline.json，供实验四对比

> 性能数字与硬件、batch size、线程数有关，本实验关注的是"同一环境下的前后对比"。
>
> 本 Notebook 已内嵌模型定义与全部工具函数，不需要导入外部 .py 文件，可直接独立运行。

In [ ]:
# ====== MobileNetV3 完整模型定义（内嵌，无需外部文件）======
import torch
import torch.nn as nn
import torch.nn.functional as F

# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

# ====== 工具函数：设备选择、算子统计、基准测试、显存统计（内嵌）======
"""算子优化实验共用工具函数。"""

import copy
import time

import torch
import torch.nn as nn


def get_device():
    """优先使用 Ascend NPU，其次 CUDA，最后 CPU。"""
    try:
        import torch_npu  # noqa: F401

        if torch.npu.is_available():
            return torch.device("npu:0")
    except Exception:
        pass

    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")


def sync_device(device):
    if device.type == "npu":
        torch.npu.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()


def count_modules(model):
    """统计模型中的 Conv2d 与 BatchNorm2d 数量。"""
    conv_count = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    bn_count = sum(1 for m in model.modules() if isinstance(m, nn.BatchNorm2d))
    return {"conv2d": conv_count, "batchnorm2d": bn_count, "total": conv_count + bn_count}


def benchmark_inference(model, x, device, warmup=5, repeats=50):
    """在指定设备上运行模型推理并返回平均时延与吞吐。"""
    model = model.to(device).eval()
    x = x.to(device)

    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        sync_device(device)

        start = time.perf_counter()
        for _ in range(repeats):
            model(x)
        sync_device(device)
        elapsed = time.perf_counter() - start

    ms_per_iter = elapsed / repeats * 1000
    throughput = repeats * x.size(0) / elapsed
    return {"ms_per_iter": ms_per_iter, "throughput": throughput, "device": str(device)}


def measure_peak_memory(fn, device):
    """运行 fn 并返回设备峰值显存（MB），CPU 上返回 None。"""
    if device.type == "npu":
        torch.npu.reset_peak_memory_stats()
        fn()
        torch.npu.synchronize()
        return torch.npu.max_memory_allocated() / 1024 ** 2
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        fn()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1024 ** 2
    return None


def fuse_conv_bn_eval(conv, bn):
    """将推理阶段的 BatchNorm 折叠进 Conv2d。"""
    fused = nn.Conv2d(
        conv.in_channels,
        conv.out_channels,
        conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        groups=conv.groups,
        bias=True,
        padding_mode=conv.padding_mode,
        device=conv.weight.device,
        dtype=conv.weight.dtype,
    )
    with torch.no_grad():
        fused.weight.data.copy_(conv.weight)
        bias = conv.bias if conv.bias is not None else torch.zeros_like(bn.bias)
        scale = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        fused.weight.data.mul_(scale.reshape(-1, 1, 1, 1))
        fused.bias.data.copy_((bias - bn.running_mean) * scale + bn.bias)
    return fused


def fuse_model(model):
    """递归折叠模型中所有 Conv+BN 结构（仅推理）。"""
    model = copy.deepcopy(model).eval()
    for name, child in model.named_children():
        if (
            isinstance(child, nn.Sequential)
            and len(child) >= 2
            and isinstance(child[0], nn.Conv2d)
            and isinstance(child[1], nn.BatchNorm2d)
        ):
            fused_conv = fuse_conv_bn_eval(child[0], child[1])
            remaining = [m for m in list(child)[2:]]
            setattr(model, name, nn.Sequential(fused_conv, *remaining))
        elif isinstance(child, nn.Module):
            setattr(model, name, fuse_model(child))
    return model

# ====== 实验主流程 ======
import json
import os
import torch

device = get_device()
print(f"设备: {device}")

model = MobileNetV3(model_mode="LARGE", num_classes=200).to(device).eval()
x = torch.randn(32, 3, 224, 224, device=device)

print("模块统计:", count_modules(model))

In [ ]:
baseline = benchmark_inference(model, x, device, warmup=5, repeats=50)

def run_once():
    with torch.no_grad():
        model(x)

baseline["peak_memory_mb"] = measure_peak_memory(run_once, device)
baseline["module_counts"] = count_modules(model)

print(f"平均时延: {baseline['ms_per_iter']:.3f} ms/iter")
print(f"吞吐: {baseline['throughput']:.1f} images/s")
if baseline["peak_memory_mb"] is not None:
    print(f"峰值显存: {baseline['peak_memory_mb']:.1f} MB")
else:
    print("CPU 环境不统计峰值显存")

In [ ]:
os.makedirs("results", exist_ok=True)
with open("results/baseline.json", "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)

print("已保存 results/baseline.json")
print(json.dumps(baseline, ensure_ascii=False, indent=2))

## 课后练习

1. (单选题) NPU 计时为什么必须在 timer.stop() 前调用 synchronize？
   - A. NPU 异步执行，需要等待 kernel 完成
   - B. 释放显存
   - C. 触发编译
   - D. 提高精度

2. (单选题) warmup 主要消除？
   - A. 首次构图、编译与缓存预热
   - B. 数据下载
   - C. 训练 loss
   - D. 模型保存

3. (多选题) 基准测试需固定？
   - A. batch_size
   - B. 输入 shape/dtype
   - C. warmup/repeats
   - D. 同步方式

4. (多选题) 峰值显存测量顺序正确的是？
   - A. reset_peak_memory_stats
   - B. 执行一次前向
   - C. synchronize
   - D. max_memory_allocated

5. (判断题) time.perf_counter() 可直接测量 NPU kernel 的真实耗时。

6. (判断题) 增大 repeats 不会降低真实耗时，但能提高统计稳定性。

7. (填空题) 读取 NPU 峰值显存的 API 是 ____。

8. (填空题) 吞吐计算：throughput = ____。

9. (简答题) 为什么首次迭代通常最慢？

10. (简答题) 如何判断数据加载是性能瓶颈？

11. (代码设计题) 编写 benchmark_forward(model, x, warmup, repeats)，返回 ms_per_iter 与 throughput。

12. (单选题) NPU 利用率低且 CPU 高，最可能？
   - A. 数据加载瓶颈
   - B. 算子计算密集
   - C. 显存不足
   - D. 模型过大

13. (多选题) results JSON 应包含？
   - A. ms_per_iter
   - B. throughput
   - C. peak_memory_mb
   - D. device/版本/输入信息

14. (判断题) baseline 与 optimized 必须使用相同输入与 repeats。

15. (简答题) 如何降低基准测量噪声？请至少给出 4 种方法。

> 参考答案见 answer/06.04_baseline_benchmark_answer.ipynb。